In [1]:
import pandas as pd
import glob
import os
import re

In [2]:
#gather all absolute file paths in raw data folder and place them in a list
def get_files():
    raw_data_path = os.path.abspath(os.path.join(os.getcwd(),".."))
    pattern = os.path.join(raw_data_path,"data","raw","*.xlsx")
    file_lst = glob.glob(pattern)
    return file_lst

In [3]:
#clean and filter data to acceptable parameters
def clean_filter(quarter_df_lst):
    #strip and force lowercase on all, force single "_" between words 
    for q in quarter_df_lst:
        q.columns = q.columns.str.lower()
        q.columns = q.columns.str.strip()
        q.columns = q.columns.str.replace('\\s+', '_', regex=True)
    #filter out any sheets with less than 5 rows
    length_check = [v for v in quarter_df_lst if len(v) > 5]
    #filter out sheets without expected cols
    required_cols = {"start","details","status_text","booking_source","player_count","tee_sheet","date_cancelled"}
    col_check = [v for v in length_check if required_cols.issubset(set(v.columns))]
    #filter out sheets with more than 10% NA values on the start time
    valid_start_time = [v for v in col_check if v["start"].isna().mean() < 0.10]
    final = valid_start_time
    return final
    
#for each file, collect each tab into a dict w/ tab name as key, df of data as value. concat all tabs into a full year df
year_dfs = []
file_lst = get_files()
for file in file_lst:
    quarter_dict = pd.read_excel(file,sheet_name=None)
    quarters = list(quarter_dict.values())
    quarters = clean_filter(quarters)
    if quarters:
        year = pd.concat(quarters)
        year_dfs.append(year)
    else: 
        print(f"'{os.path.basename(file)}': no valid data")

#concat all year dfs into one master df
master_raw = pd.concat(year_dfs,ignore_index=True)


In [4]:
print(master_raw.dtypes)

start             object
details           object
status_text       object
booking_source    object
player_count       int64
tee_sheet         object
date_cancelled    object
dtype: object


In [44]:
#basic cleaning, convert start to datetime, drop booking_source (provides no meaningful input here), rename start to tee_time for clarity
master_df = master_raw.copy()
master_df["start"] = pd.to_datetime(master_df["start"])
master_df.drop(columns=["booking_source"],inplace=True)
master_df.rename(columns={"start":"tee_time"},inplace=True)

In [46]:
#clean time, date, and cost out of details using regex,drop any rows that extract time data to nan
master_df[["book_hr","book_min","am_pm","book_month","book_day","cost_per_group"]] = master_df["details"].str.extract(r"@\s(\d{1,2}):(\d{2})(am|pm|AM|PM|Am|Pm)\s(\d{1,2})\/(\d{1,2})(?:.*?\$(\d+\.\d{2}))?")
master_df.dropna(subset=["book_hr","book_min","am_pm","book_month","book_day"],inplace=True)
master_df[["book_hr","book_min","book_month","book_day"]] = master_df[["book_hr","book_min","book_month","book_day"]].astype(int)
#reformat book time to 24hr to enable cleaning
master_df.loc[master_df["am_pm"] == "pm", "book_hr"] = master_df.loc[master_df["am_pm"] == "pm", "book_hr"]+12
master_df.loc[(master_df["book_hr"] == 12) & (master_df["am_pm"] == "am"), "book_hr"] = 0
# pull year from start, adjust booking time to previous year if booking in dec and tee time in jan
master_df["book_yr"] = master_df["tee_time"].dt.year
master_df.loc[(master_df["tee_time"].dt.month == 1)&(master_df["book_month"] == 12),"book_yr"] = master_df.loc[(master_df["tee_time"].dt.month == 1)&(master_df["book_month"] == 12),"book_yr"]-1
#convert cleaned data to datetime
master_df["booking_time"] = pd.to_datetime(
    {
        "year": master_df["book_yr"],
        "month": master_df["book_month"],
        "day": master_df["book_day"],
        "hour": master_df["book_hr"],
        "minute": master_df["book_min"],
    },
    errors="coerce",
)
# #drop helper date columns to clean up df, reorganize cols
master_df.drop(columns=["book_hr","book_min","am_pm","book_month","book_day", "book_yr"],inplace=True)
master_df = master_df[['tee_time', 'details', 'booking_time', 'status_text', 'player_count', 'tee_sheet',
       'date_cancelled', 'cost_per_group']]


display(master_df)

,tee_time,details,booking_time,status_text,player_count,tee_sheet,date_cancelled,cost_per_group
0,2021-03-14 07:51:00,Reserved using online booking @ 7:00pm 3/7 EST...,2021-03-07 19:00:00,checked in,2,Bethpage Early AM 9 Holes Blue,NaT,124.00
1,2021-03-14 07:51:00,Reserved using online booking @ 9:35pm 3/13 ES...,2021-03-13 21:35:00,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,31.00
2,2021-03-14 08:00:00,Reserved using online booking @ 3:32pm 3/13 ES...,2021-03-13 15:32:00,checked in,2,Bethpage Early AM 9 Holes Blue,NaT,62.00
3,2021-03-14 08:27:00,Reserved using online booking @ 11:19am 3/13 E...,2021-03-13 11:19:00,checked in,4,Bethpage Early AM 9 Holes Blue,NaT,124.00
4,2021-03-14 08:36:00,Reserved using online booking @ 5:42pm 3/13 ES...,2021-03-13 17:42:00,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,93.00
...,...,...,...,...,...,...,...,...
616027,2025-10-21 13:42:00,Reserved using online booking @ 8:00pm 10/14 E...,2025-10-14 20:00:00,NaN,1,Bethpage Blue Course,NaT,15.00
616028,2025-10-21 13:42:00,Reserved using online booking @ 12:32pm 10/15 ...,2025-10-16 00:32:00,NaN,1,Bethpage Blue Course,NaT,23.00
616029,2025-10-21 13:51:00,Reserved using online booking @ 7:02pm 10/14 E...,2025-10-14 19:02:00,NaN,4,Bethpage Blue Course,NaT,92.00
616030,2025-10-21 14:00:00,Reserved using online booking @ 8:55pm 10/14 EDT,2025-10-14 20:55:00,deleted,4,Bethpage Blue Course,2025-10-14 21:21:01,NaN


In [34]:
#evaluating scale of bad data to see if it makes sense to delete those records, or apply some further...
#cleaning to allow more numerical analysis
for col in master_df.columns:
    display(master_df[col].value_counts(dropna=False))
    display(master_df.loc[master_df[col].isna()])

tee_time
2024-06-03 06:39:00    47
2024-06-15 17:00:00    45
2024-06-10 06:03:00    40
2024-04-13 08:27:00    39
2024-05-23 07:06:00    39
                       ..
2022-02-16 12:39:00     1
2025-10-21 13:15:00     1
2025-10-21 12:57:00     1
2025-10-21 10:33:00     1
2025-10-21 10:24:00     1
Name: count, Length: 117823, dtype: int64

,tee_time,details,status_text,player_count,tee_sheet,date_cancelled,book_hr,book_min,am_pm,book_month,book_day,cost_per_group,book_yr,booking_time


details
Reserved using online booking @ 7:00pm 6/27 EDT Original amount due at course: 4 Players for $192.00                                 266
Reserved using online booking @ 7:00pm 5/30 EDT Original amount due at course: 4 Players for $172.00                                 238
Reserved using online booking @ 7:00pm 6/21 EDT Original amount due at course: 4 Players for $172.00                                 235
Reserved using online booking @ 7:00pm 6/20 EDT Original amount due at course: 4 Players for $172.00                                 228
Reserved using online booking @ 7:00pm 6/12 EDT Original amount due at course: 4 Players for $192.00                                 226
                                                                                                                                    ... 
Reserved using online booking @ 7:41pm 10/14 EDT Original amount due at course: 2 Players for $86.00 - PAID BOOKING FEE OF $10.00      1
Reserved using online booking @ 8

,tee_time,details,status_text,player_count,tee_sheet,date_cancelled,book_hr,book_min,am_pm,book_month,book_day,cost_per_group,book_yr,booking_time


status_text
deleted       309016
checked in    258379
teed off       35427
NaN            13206
Name: count, dtype: int64

...

,tee_time,details,status_text,player_count,tee_sheet,date_cancelled,book_hr,book_min,am_pm,book_month,book_day,cost_per_group,book_yr,booking_time
763,2021-03-23 07:42:00,Reserved using online booking @ 9:31am 3/20 ED...,checked in,1,Bethpage Blue Course,NaT,9,31,am,3,20,NaN,2021,2021-03-20 09:31:00
1270,2021-03-25 17:36:00,Reserved using online booking @ 2:57pm 3/23 ED...,deleted,1,Bethpage Green Course,2021-03-23 13:07:46,14,57,pm,3,23,NaN,2021,2021-03-23 14:57:00
3408,2021-04-06 15:39:00,Reserved using online booking @ 8:17am 4/3 EDT...,deleted,4,Bethpage Yellow 9 Hole Course,2021-04-03 07:01:59,8,17,am,4,3,NaN,2021,2021-04-03 08:17:00
3991,2021-04-08 17:09:00,Reserved using online booking @ 8:59am 4/7 EDT...,checked in,2,Bethpage Yellow 9 Hole Course,NaT,8,59,am,4,7,NaN,2021,2021-04-07 08:59:00
5056,2021-04-14 09:03:00,Reserved using online booking @ 6:15am 4/13 ED...,checked in,1,Bethpage Green Course,NaT,6,15,am,4,13,NaN,2021,2021-04-13 06:15:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615355,2025-10-13 09:03:00,Reserved using online booking @ 7:10pm 10/6 EDT,deleted,4,Bethpage Red Course,2025-10-10 17:24:01,19,10,pm,10,6,NaN,2025,2025-10-06 19:10:00
615632,2025-10-16 09:03:00,Reserved using online booking @ 2:44pm 10/15 E...,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,14,44,pm,10,15,NaN,2025,2025-10-15 14:44:00
615653,2025-10-16 11:45:00,Reserved using online booking @ 7:28pm 10/9 EDT,NaN,2,Bethpage Red Course,NaT,19,28,pm,10,9,NaN,2025,2025-10-09 19:28:00
615966,2025-10-20 13:06:00,Reserved using online booking @ 10:31pm 10/13 EDT,deleted,2,Bethpage Blue Course,2025-10-13 22:56:01,22,31,pm,10,13,NaN,2025,2025-10-13 22:31:00


book_yr
2023    169620
2024    160767
2022    155267
2021     71002
2025     59372
Name: count, dtype: int64

,tee_time,details,status_text,player_count,tee_sheet,date_cancelled,book_hr,book_min,am_pm,book_month,book_day,cost_per_group,book_yr,booking_time


booking_time
2022-05-21 19:00:00    233
2022-05-22 19:00:00    218
2022-07-10 19:00:00    216
2022-06-26 19:00:00    214
2023-06-09 19:00:00    213
                      ... 
2021-03-13 20:37:00      1
2021-03-13 20:26:00      1
2021-03-13 23:18:00      1
2021-03-13 22:55:00      1
2021-03-13 20:40:00      1
Name: count, Length: 372592, dtype: int64

,tee_time,details,status_text,player_count,tee_sheet,date_cancelled,book_hr,book_min,am_pm,book_month,book_day,cost_per_group,book_yr,booking_time
